---
format: 
    html:
      toc: true
title: "Bulk RNA Analysis"
author: "Lavinia I Fechete, Stig U Andersen, Mikkel H Schierup, Samuele Soraggi"
title-block-banner: true
---

:::{.callout-note title="Tutorial description"}

This tutorial will cover the steps for performing Differential Gene Expression on the RNA-seq data obtained from Galaxy. At the end of this tutorial you will be able to use `Python` to
    
- Perform preliminary analyses of the RNA-seq results
- Find differentially expressed genes between two conditions using `pydeseq2`

:::


**Load the necessary Python libraries**

In [ ]:
import glob
import pandas as pd
import os
from rich.pretty import pprint
import pydeseq2 as pds
from pydeseq2.dds import DeseqDataSet
from pydeseq2.utils import load_example_data  # Optional: example data

# PyDESeq2 - Data filtering and Normalization 

We will use the `pydeseq2` package to test for differentially expressed genes between our Control and Treatment Samples. You can read more about the pydeseq2 package and its functionalities [here](https://pydeseq2.readthedocs.io/en/latest/). This exercise largely follows their documentation. 


## File processing

The data for this exercise comes from the 12 tabular files with Reads per Gene counts generated by STAR Mapping in the `raw-data alignment` part of this course.
We want to create a table where each column is a sample, and the content of the table are the read counts from STAR. We must merge the 12 files with Reads per Gene information into a single file.

* If you aligned datasets in the first notebook with jupyterlab, then you will find the files - 12 files precisely - using the following command:

In [3]:
# Get the list of sample files and sort them
samples = sorted(glob.glob("results/STAR_output/*_align_contigs_1_2/*ReadsPerGene.out.tab"))
pprint(samples)

# Read and combine counts
read_counts_list = []
for sample in samples:
    df = pd.read_csv(sample, sep="\t", header=None, skiprows=4)
    read_counts_list.append(df[[1]])  # Column 2 contains unstranded read counts, similar to STAR output

# Concatenate all counts side by side (cbind equivalent)
read_counts = pd.concat(read_counts_list, axis=1)

# Optionally: set gene names from the first column of the first file
genes = list(pd.read_csv(samples[0], sep="\t", header=None, skiprows=4)[0])
read_counts.index = genes

# Optionally: assign sample names to columns
sample_names = [os.path.basename(path).replace("ReadsPerGene.out.tab", "") for path in samples]
read_counts.columns = sample_names

[
│   'results/STAR_output/S10_align_contigs_1_2/S10_1_1ReadsPerGene.out.tab',
│   'results/STAR_output/S10_align_contigs_1_2/S10_1_2ReadsPerGene.out.tab',
│   'results/STAR_output/S10_align_contigs_1_2/S10_1_3ReadsPerGene.out.tab',
│   'results/STAR_output/S10_align_contigs_1_2/S10_2_1ReadsPerGene.out.tab',
│   'results/STAR_output/S10_align_contigs_1_2/S10_2_2ReadsPerGene.out.tab',
│   'results/STAR_output/S10_align_contigs_1_2/S10_2_3ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_1_1ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_1_2ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_1_3ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_2_1ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_2_2ReadsPerGene.out.tab',
│   'results/STAR_output/TI_align_contigs_1_2/TI_2_3ReadsPerGene.out.tab'
]

In [4]:
read_counts.head()

,S10_1_1,S10_1_2,S10_1_3,S10_2_1,S10_2_2,S10_2_3,TI_1_1,TI_1_2,TI_1_3,TI_2_1,TI_2_2,TI_2_3
g175,143,226,227,217,206,193,234,205,198,221,206,165
g176,17,12,15,8,14,18,29,19,30,6,11,23
g177,0,0,0,0,0,0,0,0,0,0,0,0
g178,0,0,0,0,0,0,0,0,2,2,1,2
g179,0,0,0,0,0,0,0,0,0,0,0,0


The data frame has genes as rows and samples as columns and stores the gene expression counts (value representing the total number of sequence reads that originated from a particular gene in a sample) for each of the 12 samples. This data frame should have 12 columns and 366 rows.

In [ ]:
# Rename columns
# read_counts.columns = [
#     "S10_1_1", "S10_1_2", "S10_1_3", "S10_2_1", "S10_2_2", "S10_2_3", 
#     "TI_1_1", "TI_1_2", "TI_1_3", "TI_2_1", "TI_2_2", "TI_2_3"
# ]

# # Preview first 10 rows
# read_counts.head()

### TO SAMUELE: this column renaming already happens above. **So maybe remove the whole block?**


,S10_1_1,S10_1_2,S10_1_3,S10_2_1,S10_2_2,S10_2_3,TI_1_1,TI_1_2,TI_1_3,TI_2_1,TI_2_2,TI_2_3
g175,143,226,227,217,206,193,234,205,198,221,206,165
g176,17,12,15,8,14,18,29,19,30,6,11,23
g177,0,0,0,0,0,0,0,0,0,0,0,0
g178,0,0,0,0,0,0,0,0,2,2,1,2
g179,0,0,0,0,0,0,0,0,0,0,0,0


Import the Metadata table. This file, as its name suggests, contains information about each of the 12 RNA-seq samples, such as the treatment (Condition), genotype and replicate.

Note that the order of the rows in the Metadata table should be the same as the columns in the Read_counts file generated above.

In order to aid the following steps, we will create a Group for each sample (a new column in the metadata) based on the Genotype&Condition of each sample and assign the three replicates to this group.

In [ ]:
# Read the metadata CSV, using ";" as separator and first column as index
metadata = pd.read_csv("../Data/Clover_Data/metadata.csv", sep=";", index_col=0)

# Create 'Group' column by combining 'Genotype' and 'Condition' columns
metadata["Group"] = metadata["Genotype"].astype(str) + "_" + metadata["Condition"].astype(str)

# Display updated metadata
pprint("Metadata after adding group")
metadata

## Create the annotated data object for pydeseq2

We will merge the read counts and the metadata into a `DeseqDataSet` object, which stores the count matrix together with sample metadata and is used for all downstream analysis steps in `pydeseq2`.

Note that all the genes with zero counts across all samples were eliminated.

In [ ]:
# Optional: remove genes with all zero counts (as remove.zeros=TRUE in edgeR)
read_counts_filtered = read_counts[(read_counts > 0).sum(axis=1) > 0]

# Relevel condition so that 'Control' is the reference level
metadata["Condition"] = pd.Categorical(metadata["Condition"], categories=["Control"] + 
                                       [c for c in metadata["Condition"].unique() if c != "Control"],
                                       ordered=True)

# Create DESeq2 dataset
dds = DeseqDataSet(
    counts=read_counts_filtered.astype(int).T,    # DESeq2 expects integer counts
    metadata=metadata[["Condition", "Genotype", "Group"]],  # include relevant columns
    design="~Group",  # use Group as main design factor
)

# Check the object
dds

In [ ]:
dds.X

In [ ]:
dds.obsm

In [ ]:
dds.obsm['design_matrix']

In [ ]:
dds.obs

In [ ]:
dds.var

In [ ]:
dds.varm

## Preliminary data analysis
First, we will calculate the "pseudoCounts" as log2 of the reads per gene counts. 
<br>
This is not part of the actual differential gene expression analysis but is helpful for data exploration and quality assessment. We will look at a histogram of one of the samples and a boxplot representation of the log2 counts for all the 12 samples.
<br>
Note that there are many genes with a low number of mapped reads and that there are differences between the average read counts for each library. 

We can also create a PCA plot of the samples in order to assess the differences between the Genotypes and Conditions, but also between the replicates. In this plot the samples that are similar cluster together, while samples that are different are further apart.
<br>
In this type of plot, we would expect samples from the same group (the three replicates for each sample) to exhibit a similar gene expression profile thus clustering together while being separated from the other samples.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np


X_reduced = PCA(n_components=2).fit_transform(np.log1p(dds.X))

plt.figure()
colors = ["navy", "turquoise", "darkorange"]
lw = 2

plt.scatter( X_reduced[:,0], X_reduced[:,1], alpha=0.8, lw=1, label=dds.obs.Condition )

plt.legend(loc="best", shadow=False, scatterpoints=1)
plt.title("PCA ")


In [ ]:
X_reduced

In [ ]:
# --- R (mixOmics equivalent) ---
# resPCA <- pca(t(pseudoCounts), ncomp = 6)
# plotIndiv(resPCA, group = metadata$Genotype, pch=metadata$Condition,
#                   legend = T, legend.title = 'Genotype', legend.title.pch = 'Condition',
#                   title = 'PCA plot raw counts', style = 'ggplot2', size.xlabel = 10, size.ylabel = 10)

## Filtering the lowly expressed genes
As seen previously, many genes have a low number of read counts in our samples. The genes with very low counts across all libraries provide little evidence for differential expression, thus we should eliminate these genes before the analysis. 
<br>
In `pydeseq2`, low-count filtering is applied internally during model fitting. The basic pre-filtering step already performed above removes genes with zero counts across all samples. Additional filtering (e.g. keeping only genes with at least 10 reads in any group) can be applied manually before creating the `DeseqDataSet`.

In [ ]:
# --- R (edgeR equivalent) ---
# keep <- filterByExpr(DGEList, group=metadata$Group) #create the filter
# DGEList <- DGEList[keep, , keep.lib.sizes=FALSE] #apply the filter to on the DGEList object
# table(keep) #Check the number of genes that passed the filter

## Normalization
As we are working with multiple samples we need to normalize the read counts per gene in order to account for compositional and technical differences between the 12 RNA-seq libraries. `pydeseq2` uses the **median-of-ratios** method (the same default as DESeq2), which estimates a size factor for each sample based on the geometric mean of counts across all genes. Normalization is applied automatically when calling `dds.deseq2()`.
<br>
A size factor below one indicates that a small number of high-count genes are monopolizing the sequencing, causing the counts for other genes to be lower than would be usual given the library size.

In [ ]:
# --- R (edgeR equivalent) ---
# DGEList <- calcNormFactors(DGEList, method="RLE")
# DGEList$samples

## Normalized counts - Exploratory Data analysis
For data analysis purposes normalized log2 counts can be extracted from the DGEList object using the function CPM (counts per million).

We will generate the same plots as for the raw counts in order to compare the data before and after normalization. 

Do the plots for normalized counts look different compared with the plots computed before data filtering and normalization? 

In [ ]:
# --- R (edgeR / mixOmics equivalent) ---
# pseudoNormCounts <- cpm(DGEList, log = TRUE, prior.count = 1)
# head(pseudoNormCounts)
# hist(pseudoNormCounts[ ,"S10_1_1"], main = "", xlab = "counts")
# boxplot(pseudoNormCounts, col = "gray", las = 3, cex.names = 1)
#
# resPCA <- pca(t(pseudoNormCounts), ncomp = 6)
# plotIndiv(resPCA, group = metadata$Genotype, pch=metadata$Condition,
#                   legend = T, legend.title = 'Genotype', legend.title.pch = 'Condition',
#                   title = 'PCA plot normalized counts', style = 'ggplot2', size.xlabel = 10, size.ylabel = 10)

# PyDESeq2 - Testing for Differentially expressed genes (DEGs)

We use `pydeseq2` for Differential Gene Expression using a Generalized Linear Model (GLM) with a negative binomial distribution, accounting for treatment and genotype effects.


After we concluded the exploratory data analysis and the filtering and normalization steps we can now start testing for differentially expressed genes between our samples. 


In `pydeseq2`, the design matrix is constructed automatically from the `design` formula specified when creating the `DeseqDataSet` (e.g. `"~Group"`). You can inspect it via `dds.obsm['design_matrix']`. The samples belonging to each group are assigned indicator values in this matrix.

In [ ]:
# --- R (edgeR equivalent) ---
# design.matrix <- model.matrix(~0+Group)
# rownames(design.matrix) <- colnames(DGEList)
# colnames(design.matrix) <- levels(metadata$Group)
# design.matrix

For pairwise comparisons between groups, `pydeseq2` uses the `contrast` parameter in `DeseqStats`, which directly specifies the factor name, numerator level, and denominator level. For example:

- `contrast=["Group", "S10_Treatment", "S10_Control"]` tests S10 Treatment vs S10 Control  
- `contrast=["Group", "Tienshan_Treatment", "Tienshan_Control"]` tests Tienshan Treatment vs Tienshan Control  

The combined analysis across both genotypes is done with a separate `DeseqDataSet` using `design="~Condition"`.

In [ ]:
# --- R (edgeR equivalent) ---
# contrasts <- makeContrasts(
#                 S10 = S10_Treatment-S10_Control,
#                 Tienshan = Tienshan_Treatment-Tienshan_Control,
#                 S10_Tienshan= (S10_Treatment+Tienshan_Treatment)/2 - (S10_Control+Tienshan_Control)/2,
#                 levels=design.matrix)
# contrasts

In [ ]:
from pydeseq2.ds import DeseqStats

# Run DESeq2 model
dds.deseq2()

# Set up stats object with desired contrast
stat_res = DeseqStats(dds, contrast=["Group", "Tienshan_Treatment", "Tienshan_Control"])
stat_res.summary()
res_df_Ti = stat_res.results_df

In [ ]:
stat_res.contrast

In [ ]:
res_df_Ti

*   *Any ideas for other contrasts that migth be interesting to explore?*







The following cell shows the edgeR equivalent steps (commented out). In `pydeseq2`, dispersion estimation and model fitting are both handled automatically inside `dds.deseq2()`, which was already called above.

In [ ]:
# --- R (edgeR equivalent) ---
# DGEList <- estimateDisp(DGEList, design.matrix)
# fit <- glmQLFit(DGEList, design.matrix)

## DEGs for White clover S10 plants Treatment vs Control

Find DEGs for White clover S10 plants in Treatment condition compared with the Control condition.
We test for differentially expressed genes using `DeseqStats` from `pydeseq2`, which applies a Wald test on the negative binomial GLM coefficients. We specify the contrast of interest and call `summary()` to extract results for all genes.

In [ ]:
# --- R (edgeR equivalent) ---
# glmqlf_S10 <- glmQLFTest(fit, contrast=contrasts[,"S10"])
# DEG_S10 <- topTags(glmqlf_S10, n = nrow(DGEList$counts))
# DEG_S10

In [ ]:
# Perform statistical test for S10_Treatment vs S10_Control
stat_S10 = DeseqStats(dds, contrast=["Group", "S10_Treatment", "S10_Control"], )
stat_S10.summary()

# Extract results sorted by adjusted p-value (like topTags)
deg_S10 = stat_S10.results_df.sort_values("padj")


We have now created a table with certain parameters calculated for each of the genes analysed.

**log2FoldChange** represents the base 2 logarithm of the fold change and shows how much the expression of the gene has changed between the two conditions. A log2FoldChange of 1 means a doubling in the read count between control and treatment samples.
Genes with a log2FoldChange higher than 0 are upregulated while genes with a log2FoldChange lower than 0 are downregulated.
<br>
**baseMean** represents the average normalized count across all samples, reflecting the overall expression level of the gene.

**stat** - the Wald test statistic.

**pvalue** is the raw p-value.

**padj** (The adjusted p-value) is calculated using Benjamini and Hochberg's algorithm. It controls the rate of false positive values under multiple testing. Usually, a threshold of under 5% is set for the adjusted p-value.

The important information for us in this table is stored in the **log2FoldChange** and **padj** columns. The top DE genes have small padj values and large fold changes.

Many of the genes in the samples are uninteresting for us, as they have a high FDR and/or low logFC values so we cannot consider them as differentially expressed.

We will apply a filtering step in order to keep only the statistically significant genes. We will filter out the genes with an FDR higher than 0.05 and an absolute logFC lower than 1.

In [ ]:
# --- R (edgeR equivalent) ---
# DEG_S10_filtered <- DEG_S10$table[DEG_S10$table$FDR < 0.05 & abs(DEG_S10$table$logFC) > 1,]
# DEG_S10_filtered <- rownames_to_column(DEG_S10_filtered) %>% rename(gene_ID = rowname)
# head(DEG_S10_filtered)
# nrow(DEG_S10_filtered)

In [ ]:
# Filter by adjusted p-value and log2 fold change
deg_S10_filtered = deg_S10[(deg_S10["padj"] < 0.05) & (deg_S10["log2FoldChange"].abs() > 1)].copy()

# Add gene_ID column from index (like rownames_to_column)
deg_S10_filtered.reset_index(inplace=True)
deg_S10_filtered.rename(columns={"index": "gene_ID"}, inplace=True)

# Preview top genes
print(deg_S10_filtered.head())

# Number of genes that passed the filter
print(len(deg_S10_filtered))


We can also visualize the selected genes by plotting a Smear plot or a Volcano plot. The Genes that passed the filter are coloured in red, and the top 10 genes with the lowest FDR value are labelled with their gene ID.

We can see that the majority of the genes analysed are either not statistically significant or have a very small logFC.

In [ ]:
# --- R (edgeR equivalent) ---
# plotSmear(glmqlf_S10,
#           de.tags = rownames(DEG_S10$table)[which(DEG_S10$table$FDR < 0.05 & abs(DEG_S10$table$logFC) > 1)])
# text(x=DEG_S10_filtered$logCPM[1:10],
#      y=DEG_S10_filtered$logFC[1:10],
#      labels=DEG_S10_filtered$gene_ID[1:10], cex=0.7, pos=1)
# abline(h = c(-1, 1), col = "blue")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Scatter plot of all genes: logCPM vs log2FC
plt.figure(figsize=(10, 6))
plt.scatter(deg_S10["baseMean"].apply(lambda x: np.log2(x + 1)), 
            deg_S10["log2FoldChange"], 
            color="grey", alpha=0.4, label="All genes")

# Highlight filtered DEGs in red
plt.scatter(deg_S10_filtered["baseMean"].apply(lambda x: np.log2(x + 1)), 
            deg_S10_filtered["log2FoldChange"], 
            color="red", alpha=0.8, label="DEGs (FDR < 0.05, |LFC| > 1)")

# Label top 10 DEGs
for i in range(min(10, len(deg_S10_filtered))):
    x = np.log2(deg_S10_filtered.loc[i, "baseMean"] + 1)
    y = deg_S10_filtered.loc[i, "log2FoldChange"]
    label = deg_S10_filtered.iloc[i, 0]
    plt.text(x, y, label, fontsize=8, ha='center', va='bottom')

# Add horizontal lines at logFC ±1
plt.axhline(y=1, color="blue", linestyle="--")
plt.axhline(y=-1, color="blue", linestyle="--")

plt.xlabel("log2(baseMean + 1) ~ logCPM")
plt.ylabel("log2 Fold Change")
plt.title("Smear Plot of S10 DEGs")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Prepare data for plotting
deg_S10["logCPM"] = np.log2(deg_S10["baseMean"] + 1)  # Approximate logCPM
deg_S10_filtered["logCPM"] = np.log2(deg_S10_filtered["baseMean"] + 1)

# Create Plotly scatter plot
fig = px.scatter(
    deg_S10, 
    x="logCPM", 
    y="log2FoldChange", 
    color=(deg_S10["padj"] < 0.05) & (deg_S10["log2FoldChange"].abs() > 1),
    title="Smear Plot of S10 DEGs",
    labels={"logCPM": "log2(baseMean + 1)", "log2FoldChange": "log2 Fold Change"},
    color_continuous_scale=["grey", "red"],
    opacity=0.7
)

# Add the top 10 gene labels
for i in range(min(10, len(deg_S10_filtered))):
    fig.add_annotation(
        x=deg_S10_filtered.iloc[i]["logCPM"],
        y=deg_S10_filtered.iloc[i]["log2FoldChange"],
        text=deg_S10_filtered.iloc[i][0],
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-40,
        font=dict(size=8)
    )

# Add horizontal lines at logFC ±1
fig.add_hline(y=1, line_dash="dash", line_color="blue")
fig.add_hline(y=-1, line_dash="dash", line_color="blue")

# Show the figure
fig.show()


We can also create a heatmap with the log2 read counts of the selected differentially expressed genes so that we can visualise the differences in normalized counts between the Control and the Treatment samples.

The genes are ordered by the FDR value. 

In [ ]:
# --- R (pheatmap equivalent) ---
# annot_col <- data.frame(row.names = colnames(pseudoNormCounts)[1:6], Condition = c(rep("Control", 3),rep( "Treatment", 3)))
# pheatmap(as.matrix(pseudoNormCounts[DEG_S10_filtered$gene_ID,c(1:6)]), cluster_rows = F, cluster_col = F, annotation_col = annot_col)

In [ ]:
heatmap_counts.index = list(heatmap_counts.index)

In [ ]:
heatmap_counts.columns.shape

In [ ]:
# Assuming your dataframe is named df
df_reset = heatmap_counts.set_index(heatmap_counts.index[heatmap_counts.index.str.startswith('g')])

# Now, df_reset will have a 1D index
print(df_reset.head())

In [ ]:
dds.var_names

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select the genes that passed the filtering (DEGs) from the pseudo-normalized counts
selected_genes = deg_S10_filtered.gene_ID.values
heatmap_counts = pd.DataFrame(pds.preprocessing.deseq2_norm(dds.X)[0][:6,:], index=dds.obs_names[:6], columns=dds.var_names)
heatmap_counts =  heatmap_counts[selected_genes].T   # select first 6 samples
annot_col = pd.DataFrame({"Condition": ["Control"]*3 + ["Treatment"]*3}, index=dds.obs_names[:6])

In [ ]:
heatmap_counts.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select the genes that passed the filtering (DEGs) from the pseudo-normalized counts
import seaborn as sns
import matplotlib.pyplot as plt

# Select the genes that passed the filtering (DEGs) from the pseudo-normalized counts
selected_genes = deg_S10_filtered.gene_ID.values
heatmap_counts = pd.DataFrame(pds.preprocessing.deseq2_norm(dds.X)[0][:6,:], index=dds.obs_names[:6], columns=dds.var_names)
heatmap_counts =  heatmap_counts[selected_genes].T   # select first 6 samples
annot_col = pd.DataFrame({"Condition": ["Control"]*3 + ["Treatment"]*3}, index=dds.obs_names[:6])

# Create the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(np.log1p(heatmap_counts), 
            cmap="coolwarm",  # Change color scheme as needed
            annot=False,       # Don't annotate each cell (you can enable this if necessary)
            cbar_kws={'label': 'Normalized Counts (log(x+1))'},
            xticklabels=annot_col["Condition"],
            yticklabels=selected_genes,  # Display gene IDs on the y-axis
            square=True,       # Optional: square grid
            linewidths=0.5,    # Optional: separate cells with lines
            linecolor="black")

# Add title and labels
plt.title("Heatmap of DEGs (Filtered) for S10", fontsize=14)
plt.xlabel("Condition (Control vs Treatment)")
plt.ylabel("Genes")

plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px


# Select the genes that passed the filtering (DEGs) from the pseudo-normalized counts
selected_genes = deg_S10_filtered.gene_ID.values
heatmap_counts = pd.DataFrame(pds.preprocessing.deseq2_norm(dds.X)[0][:6,:], index=dds.obs_names[:6], columns=dds.var_names)
heatmap_counts =  heatmap_counts[selected_genes].T   # select first 6 samples
annot_col = pd.DataFrame({"Condition": ["Control"]*3 + ["Treatment"]*3}, index=dds.obs_names[:6])


# Create a heatmap
fig = px.imshow(np.log1p(heatmap_counts),
                labels={'x': "Samples", 'y': "Genes", 'color': 'Normalized Counts'},
                x=heatmap_counts.columns, 
                y=selected_genes,
                color_continuous_scale='Viridis')

fig.update_layout(
    title="Heatmap of DEGs (Filtered) for S10",
    height=1000,    # Increase overall height
    width=1800,     # Increase overall width
    autosize=True,  # Auto-sizing to fit the data
    xaxis=dict(
        scaleanchor="y",  # Keeps the aspect ratio square
        tickangle=45,     # Optional: rotate axis labels for better visibility
    ),
#    yaxis=dict(
#        scaleanchor="x",  # Keeps the aspect ratio square
#    ),
    margin=dict(t=50, r=50, b=50, l=50)  # Optional: add margins for better spacing
)


# Add annotations for the conditions (Control vs Treatment)
for i, col in enumerate(heatmap_counts.columns):
    fig.add_annotation(
        x=i,
        y=-8,
        text=annot_col["Condition"].iloc[i],
        showarrow=False,
        font=dict(size=12),
        textangle=90,    
        align="center"
    )


fig.update_layout(title="Heatmap of DEGs (Filtered) for S10")#, height=600, width=800)
fig.show()


## DEGs for White clover Tienshan Treatment vs Control
We can do this using the same functions as above and changing the contrast.

This time we will directly filter the differentially expressed genes using the same parameters as for the S10 samples.

In [ ]:
# --- R (edgeR equivalent) ---
# glmqlf_Ti <- glmQLFTest(fit, contrast=contrasts[,"Tienshan"])
# DEG_Ti <- topTags(glmqlf_Ti, n = nrow(DGEList$counts))
# DEG_Ti_filtered <- DEG_Ti$table[DEG_Ti$table$FDR < 0.05 & abs(DEG_Ti$table$logFC) > 1,]
# DEG_Ti_filtered <- rownames_to_column(DEG_Ti_filtered) %>% rename(gene_ID = rowname)
# print(head(DEG_Ti_filtered))
# print("Nr of differentially expressed genes:")
# print(nrow(DEG_Ti_filtered))

In [ ]:
import pandas as pd

# Filter by FDR < 0.05 and absolute log2FC > 1
DEG_Ti_filtered = res_df_Ti[(res_df_Ti["padj"] < 0.05) & (res_df_Ti["log2FoldChange"].abs() > 1)].copy()

# Reset index to expose gene IDs as a column
DEG_Ti_filtered = DEG_Ti_filtered.reset_index().rename(columns={"index": "gene_ID"})

# Show first few rows
print(DEG_Ti_filtered.head())

# Print number of DEGs
print("Nr of differentially expressed genes:")
print(len(DEG_Ti_filtered))


We can now plot a Venn diagram with the DEGs for the two genotypes in order to observe the number of common and specific differentially expressed genes between the two genotypes as response to the cold exposure. You can see that a high percentage of the identified genes are common for the two genotypes, while each genotype has also specific genes.

In [ ]:
# --- R (VennDiagram equivalent) ---
# vd <- venn.diagram(
#   x = list(deg_S10_filtered$gene_ID, DEG_Ti_filtered$gene_ID),
#   category.names = c("S10" , "Tienshan"),
#   lwd = 4,
#   fill = c("cornflowerblue", "yellowgreen"),
#   filename = NULL,
#   cat.cex = 1,
#   cat.fontface = "bold",
#   output=TRUE
# )
# grid.draw(vd)

In [ ]:
!pip install matplotlib_venn

In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt

# Assuming DEG_S10_filtered and DEG_Ti_filtered are pandas DataFrames with a 'gene_ID' column
set_S10 = set(deg_S10_filtered['gene_ID'])
set_Ti = set(DEG_Ti_filtered['gene_ID'])

# Create the Venn diagram
plt.figure(figsize=(6,6))
venn = venn2([set_S10, set_Ti], 
             set_labels=('S10', 'Tienshan'),
             set_colors=('cornflowerblue', 'yellowgreen'),
             alpha=0.7)

# Bold category labels
for text in venn.set_labels:
    text.set_fontweight('bold')
    text.set_fontsize(12)

# Thicker lines
for patch in venn.patches:
    if patch:
        patch.set_linewidth(2)

plt.title("Venn Diagram of DEG overlap", fontsize=14)
plt.show()


## DEGs for S10+Tienshan Treatment vs Control w/o genotype effects

Until now we tested for DEGs specific for each of the two genotypes under cold treatment. We can also run a test where we ignore the genotype and just test for the differences in the cold response.
<br>
Consider the counts for both genotypes as a single dataset using the previously created contrast "S10_Tienshan". 
<br>
* Do the results look different compared with the previous tests?

In [ ]:
# Create DESeq2 dataset
dds2 = DeseqDataSet(
    counts=read_counts_filtered.astype(int).T,    # DESeq2 expects integer counts
    metadata=metadata[["Condition", "Genotype", "Group"]],  # include relevant columns
    design="~Condition",  # use Group as main design factor
)

In [ ]:
dds2.obs

In [ ]:
dds2.obsm['design_matrix']

In [ ]:
dds2.deseq2()

# Perform statistical test for S10_Treatment vs S10_Control
stat_genotype = DeseqStats(dds2, contrast=["Condition", "Treatment", "Control"], )
stat_genotype.summary()

# Extract results sorted by adjusted p-value (like topTags)
deg_genotype = stat_genotype.results_df.sort_values("padj")


In [ ]:
# Filter by FDR < 0.05 and absolute log2FC > 1
DEG_cond_filtered = deg_genotype[( deg_genotype["padj"] < 0.05) & ( deg_genotype["log2FoldChange"].abs() > 1)].copy()

# Reset index to expose gene IDs as a column
DEG_cond_filtered = DEG_cond_filtered.reset_index().rename(columns={"index": "gene_ID"})

# Show first few rows
print(DEG_cond_filtered.head())

# Print number of DEGs
print("Nr of differentially expressed genes:")
print(len(DEG_cond_filtered))


In [ ]:
#until here

In [ ]:
# --- R (edgeR equivalent) ---
# glmqlf_S10_Ti <- glmQLFTest(fit, contrast=contrasts[,"S10_Tienshan"])
# DEG_S10_Ti <- topTags(glmqlf_S10_Ti, n = nrow(DGEList$counts))
# DEG_S10_Ti_filtered <- DEG_S10_Ti$table[DEG_S10_Ti$table$FDR < 0.05 & abs(DEG_S10_Ti$table$logFC) > 1,]
# DEG_S10_Ti_filtered <- rownames_to_column(DEG_S10_Ti_filtered) %>% rename(gene_ID = rowname)
# print(head(DEG_S10_Ti_filtered))
# print("Nr of differentially expressed genes:")
# print(nrow(DEG_S10_Ti_filtered))

Plot the results from the 3 tests in a Venn diagram to visualize the number of common and unique genes

In [ ]:
# --- R (VennDiagram equivalent) ---
# vd <- venn.diagram(
#   x = list(DEG_S10_filtered$gene_ID, DEG_Ti_filtered$gene_ID,
#            DEG_S10_Ti_filtered$gene_ID),
#   category.names = c("S10" , "Tienshan", "S10_Tienshan"),
#   lwd = 3,
#   fill = c("cornflowerblue", "yellowgreen", "thistle3"),
#   filename = NULL,
#   cat.cex = 1,
#   cat.fontface = "bold",
#   output=TRUE
# )
# grid.draw(vd)

In [ ]:
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

set_S10 = set(deg_S10_filtered['gene_ID'])
set_Ti = set(DEG_Ti_filtered['gene_ID'])
set_cond = set(DEG_cond_filtered['gene_ID'])

plt.figure(figsize=(7, 7))
venn = venn3(
    [set_S10, set_Ti, set_cond],
    set_labels=('S10', 'Tienshan', 'S10+Tienshan'),
    set_colors=('cornflowerblue', 'yellowgreen', 'thistle'),
    alpha=0.7
)
for text in venn.set_labels:
    text.set_fontweight('bold')
    text.set_fontsize(12)
plt.title("Venn Diagram of DEG overlap (3 contrasts)", fontsize=14)
plt.show()

## Explore DGE results

### Select the genes which appear only in the analysis using both genotypes for further examination.

We use pandas boolean indexing to keep only the genes that are unique to the combined analysis — i.e., not present in either the S10-only or Tienshan-only results.

In [ ]:
# Keep only genes unique to the combined analysis (not found in S10-only or Ti-only results)
S10_Ti_unique = DEG_cond_filtered[
    ~DEG_cond_filtered['gene_ID'].isin(deg_S10_filtered['gene_ID']) &
    ~DEG_cond_filtered['gene_ID'].isin(DEG_Ti_filtered['gene_ID'])
].copy()
S10_Ti_unique

In [ ]:
# --- R (dplyr equivalent) ---
# S10_Ti_unique <- anti_join(DEG_S10_Ti_filtered, DEG_S10_filtered, by="gene_ID") %>%
#                        anti_join(DEG_Ti_filtered, by="gene_ID")
# S10_Ti_unique

In [ ]:
read_counts_reset = read_counts.reset_index().rename(columns={"index": "gene_ID"})
S10_Ti_unique_counts = S10_Ti_unique[['gene_ID', 'log2FoldChange', 'padj']].merge(
    read_counts_reset, on='gene_ID'
)
S10_Ti_unique_counts

### Linking the genes selected as differentially expressed back to the raw read counts

How do the counts look for these genes, does it make sense that they are differentially expressed only when using the two genotypes?

In [ ]:
Functional_annotations = pd.read_csv("../Data/Clover_Data/Functional_Annotations.txt", sep="\t")
S10_Ti_unique_FA = S10_Ti_unique[['gene_ID', 'log2FoldChange', 'padj']].merge(
    Functional_annotations, on='gene_ID'
)
S10_Ti_unique_FA

In [ ]:
# --- R (dplyr equivalent) ---
# Read_counts <- rownames_to_column(Read_counts) %>% rename(gene_ID = rowname)
# S10_Ti_unique_counts <- inner_join(S10_Ti_unique[,c(1,2,6)], Read_counts, by="gene_ID")
# S10_Ti_unique_counts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(deg_S10['padj'].dropna(), bins=200, range=(0, 0.1), color='steelblue', edgecolor='black')
plt.xlabel("padj (adjusted p-value)")
plt.title("")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(deg_S10['log2FoldChange'].dropna(), bins=50, range=(-6, 6), color='steelblue', edgecolor='black')
plt.xlabel("log2FoldChange")
plt.title("")
plt.tight_layout()
plt.show()

### Adding functional annotation to DEGs
Until now we looked only at gene IDs, but we can also add functional annotations to the DEGs. The functional annotations were generated using protein sequences and the `EggNOG` software.

Identifying the molecular function of the differentially expressed genes can help us do a literature survey in order to check if any of the genes discovered have been previously identified as being involved in the cold response.

In [ ]:
import os
os.makedirs("DEG_Output_tables", exist_ok=True)

In [ ]:
read_counts_reset = read_counts.reset_index().rename(columns={"index": "gene_ID"})
ti_cols = ['gene_ID'] + list(read_counts.columns[6:12])  # Ti samples: TI_1_1 through TI_2_3

DEG_Ti_counts_FA = (
    DEG_Ti_filtered[['gene_ID', 'log2FoldChange', 'padj']]
    .merge(read_counts_reset[ti_cols], on='gene_ID')
    .merge(Functional_annotations, on='gene_ID')
)

DEG_Ti_counts_FA.to_csv(
    "DEG_Output_tables/Ti_Treatment_Control_DGE.txt",
    sep="\t", index=False
)

DEG_Ti_counts_FA.head(10)

In [ ]:
# --- R (dplyr equivalent) ---
# Functional_annotations <- read.delim("../Data/Clover_Data/Functional_Annotations.txt")
# S10_Ti_unique_FA <- inner_join(S10_Ti_unique[,c(1,2,6)], Functional_annotations, by="gene_ID")
# S10_Ti_unique_FA

In [ ]:
DEG_Ti_up = DEG_Ti_counts_FA[DEG_Ti_counts_FA['log2FoldChange'] > 0]
print("Nr of upregulated genes:")
print(len(DEG_Ti_up))

DEG_Ti_down = DEG_Ti_counts_FA[DEG_Ti_counts_FA['log2FoldChange'] < 0]
print("Nr of downregulated genes:")
print(len(DEG_Ti_down))

:::{.callout-tip title="Task:"}
*   Based on the results obtained in the analysis so far, would you change the cut-off for the FDR and logFC to be more strict or more permissive? Look back at the raw counts for different FDR and logFC values and set the thresholds as you find appropriate.

You can also plot histograms with the FDR and logFC values. 





In [ ]:
# --- R equivalent ---
# hist(DEG_S10$table$FDR , main = "", xlab = "FDR",  breaks= 200, xlim = range(c(0, 0.1)))

In [ ]:
# --- R equivalent ---
# hist(DEG_S10$table$logFC , main = "", xlab = "logFC",  breaks= 50, xlim = range(c(-6, 6)))

* Separate the upregulated and downregulated genes for each genotype and append functional annotations to them. 
* Identify the genes that are commonly upregulated in S10 and Tienshan samples and the uniquely upregulated genes for each genotype.
* Why do you think some of the proteins appear in duplicates? <br>

To answer these questions, it may be convenient to save summary tables from R and open them in excel. See the code below for examples of how to do this. Files can be downloaded by right-clicking on the file name.

If you are familiar with R functions, you are welcome to use those for counting.


In [ ]:
# --- R equivalent ---
# dir.create("DEG_Output_tables", showWarnings = FALSE)

In [ ]:
# --- R (dplyr equivalent) ---
# #Create the table with the DEGs, Raw counts(just for the Ti samples in this case, change to columns (2:7) for the S10 samples) and Functional annotations
# DEG_Ti_counts_FA <- inner_join(DEG_Ti_filtered[,c(1, 2, 6)], Read_counts[, c(1, 8:13)], by="gene_ID") %>%
#                  inner_join(Functional_annotations, by="gene_ID")
# #Write the table to file
# write.table(DEG_Ti_counts_FA, file = "DEG_Output_tables/Ti_Treatment_Control_DGE.txt", quote = FALSE, row.names = FALSE, sep = "\t")
# #Display the first 10 rows of the table
# head(DEG_Ti_counts_FA, n=10)

Example for filtering and counting the Up/Down genes. You can easily filter using pandas boolean indexing on the `log2FoldChange` column.

In [ ]:
# --- R (dplyr equivalent) ---
# DEG_Ti_up <- filter(DEG_Ti_counts_FA, logFC > 0)
# print("Nr of upregulated genes:")
# print(nrow(DEG_Ti_up))
# DEG_Ti_down <- filter(DEG_Ti_counts_FA, logFC < 0)
# print("Nr of downregulated genes:")
# print(nrow(DEG_Ti_down))

You can use the "inner_join" and the "anti_join" functions from the dplyr package to select the common and unique genes for each genotype:

`example_file_joined <- inner_join(file1, file2, by="gene_ID")`
:::

**End of the task**

---

:::{.callout-note title="Wrapping up"}

In this notebook, you have worked with RNA-seq results for two white clover genotypes exposed to one night of cold treatment, aiming to identify genes that change their expression in response to the cold treatment. You have learned to perform exploratory data analysis of raw and normalized RNA-seq read counts. You have also performed differential gene expression using `pydeseq2`. 

Do you want to look for more analysis? We have an introductory course on bulkRNA analysis at the Danish Health Data Science Sandbox (held periodically in Copenhagen, [keep an eye on the webpage where they list courses](https://heads.ku.dk/course/)). You can also find the material on the  **Transcriptomics Sandbox** on uCloud (for danish users of uCloud), otherwise the course is documented in web format at the respective webpage.

&nbsp;

 <p align="center">
  <a href="https://cloud.sdu.dk/app/jobs/create?app=transcriptomics" style="background-color: #4266A1; color: #FFFFFF; padding: 20px 20px; text-decoration: none; border-radius: 5px;">
    Transcriptomics Sandbox
  </a>
</p>
  <br>
  <br>

<p align="center">
  <a href="https://hds-sandbox.github.io/bulk_RNAseq_course/" style="background-color: #4266A1; color: #FFFFFF; padding: 20px 20px; text-decoration: none; border-radius: 5px;">
    bulkRNA analysis course
  </a>
</p>
  <br>
  <br>

:::